# 08c · Is the transformer better than our champion? BankSight

**Question:** do the fine-tuned transformers (trained on Colab in 08b) predict relief better than our champion model B4 (TF-IDF + logistic regression)?

**What we compare (all on the same validation complaints):**
* **B1** relief rate of the product (the floor)
* **B4** our champion
* **each transformer**  trained in 08b (DistilBERT, and RoBERTa if you ran it), using epoch 2
* **Blend** the average rank of B4 and a transformer (an ensemble, labelled *exploratory* because we try it only now)



**What this does:** loads the libraries.

In [13]:
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import display
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score, roc_auc_score

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

**What this does:** sets file locations and the settings of our rule. `EPOCH_TO_USE = 2` was decided in advance.

In [14]:
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
WAREHOUSE = ROOT / "data" / "warehouse" / "banksight.duckdb"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports" / "transformer"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TRANSFORMER_TAGS = ["distilbert", "roberta"]   # the models you trained in 08b; any whose prediction file is missing is skipped
EPOCH_TO_USE = 2
MIN_GAIN = 0.01
N_BOOT = 300
SEED = 42
MIN_N_AUC = 300
OPENING_CHARS = 150
MIN_REPEATS = 5

PRED_DIR = ROOT / "data" / "colab" / "outputs"
assert (MODEL_DIR / "04_champion_bundle.joblib").exists() and (MODEL_DIR / "07_topic_model.joblib").exists(), "Run notebooks 04 and 07 first."

## 0. Load the data, B4's scores and the transformers' scores

**What this does:** reads train and validation complaints (test excluded in the SQL) and cleans the text exactly as in notebooks 04 and 07.

In [15]:
con = duckdb.connect(str(WAREHOUSE), read_only=True)
df = con.sql("""
    SELECT l.complaint_id, l.date_received, l.product_group, l.got_relief, l.split, r.issue, r.narrative
    FROM features.complaint_labels l
    JOIN raw.cfpb_complaints r USING (complaint_id)
    WHERE l.target_eligible
      AND l.split IN ('train', 'validation')
      AND r.narrative IS NOT NULL AND length(trim(r.narrative)) > 0
""").df()
con.close()

df["issue"] = df["issue"].fillna("Unknown").astype(str)
df["date_received"] = pd.to_datetime(df["date_received"])
df = df.sort_values("date_received", kind="stable").reset_index(drop=True)

clean_text = df["narrative"].str.replace(r"\{\$[\d,\.]+\}|\$\s?[\d,]+(?:\.\d+)?", " amtmask ", regex=True)
clean_text = clean_text.str.replace(r"\bXX/XX/(?:XXXX|\d{2,4})\b|\bXX/\d{2}/\d{2,4}\b", " datemask ", regex=True)
clean_text = clean_text.str.replace(r"\bX{2,}\b", " xmask ", regex=True)
df["text"] = clean_text.str.replace(r"\s+", " ", regex=True).str.strip()
is_valid = (df["split"] == "validation").to_numpy()
print(f"rows {len(df):,} | validation rows {is_valid.sum():,}")

rows 122,314 | validation rows 18,537


**What this does:** scores validation with B4 (from notebook 04) and with the product-only baseline B1.

In [16]:
bundle = joblib.load(MODEL_DIR / "04_champion_bundle.joblib")
valid = df[is_valid].copy()
X_valid = sp.hstack([bundle["tfidf"].transform(valid["text"]), bundle["ohe"].transform(valid[["product_group", "issue"]])]).tocsr()
valid["p_b4"] = bundle["b4_model"].predict_proba(X_valid)[:, 1]

product_rate_train = df[~is_valid].groupby("product_group")["got_relief"].mean()
valid["p_b1"] = valid["product_group"].map(product_rate_train).fillna(df[~is_valid]["got_relief"].mean())

**What this does:** attaches each transformer's probabilities (matched on complaint id, so no row can be mixed up) and builds one blend per transformer.
Rank-averaging means each model's ordering counts equally, whatever scale its probabilities use.

In [17]:
valid["complaint_id"] = valid["complaint_id"].astype(str)
found_tags = []

for tag in TRANSFORMER_TAGS:
    pred_file = PRED_DIR / f"val_predictions_{tag}_epoch{EPOCH_TO_USE}.csv"
    if not pred_file.exists():
        print(f"skipping {tag}: {pred_file.name} not found in {PRED_DIR}")
        continue
    preds = pd.read_csv(pred_file)
    preds["complaint_id"] = preds["complaint_id"].astype(str)
    preds = preds.rename(columns={"p_relief": f"p_{tag}"})
    valid = valid.merge(preds, on="complaint_id", how="left")
    assert valid[f"p_{tag}"].notna().all(), f"Some validation complaints have no {tag} prediction. Re-run 08b so it covers every validation row."
    valid[f"p_blend_{tag}"] = (rankdata(valid["p_b4"]) + rankdata(valid[f"p_{tag}"])) / (2 * len(valid))
    found_tags.append(tag)
    print("loaded", tag)

assert len(found_tags) > 0, "No prediction files found. Copy them from Google Drive (banksight_colab/outputs) into data/colab/outputs/ first."
y = valid["got_relief"].to_numpy()
print(f"validation rows {len(valid):,} | relief rate {y.mean():.3f}")

loaded distilbert
loaded roberta
validation rows 18,537 | relief rate 0.228


**What this does:** puts the four score columns in one dictionary so the next cells can loop over them.

In [18]:
model_columns = {"B1 product rate": "p_b1", "B4 champion": "p_b4"}
for tag in found_tags:
    model_columns[f"{tag} (transformer)"] = f"p_{tag}"
    model_columns[f"blend B4 + {tag}"] = f"p_blend_{tag}"

print(list(model_columns.keys()))

['B1 product rate', 'B4 champion', 'distilbert (transformer)', 'blend B4 + distilbert', 'roberta (transformer)', 'blend B4 + roberta']


## 1. Overall comparison
PR-AUC is the headline. **Within-product ROC-AUC** asks the harder question: can the model tell apart complaints *inside* the same product?

**What this does:** computes PR-AUC, ROC-AUC and within-product ROC-AUC (an average over products with at least 100 complaints, weighted by size) for each model.

In [19]:
rows = []
for model_name, column in model_columns.items():
    scores = valid[column].to_numpy()

    within_values = []
    within_weights = []
    for product, part in valid.groupby("product_group"):
        y_part = part["got_relief"].to_numpy()
        if len(part) >= 100 and 0 < y_part.sum() < len(part):
            within_values.append(roc_auc_score(y_part, part[column]))
            within_weights.append(len(part))

    rows.append({
        "model": model_name,
        "pr_auc": average_precision_score(y, scores),
        "roc_auc": roc_auc_score(y, scores),
        "within_product_roc_auc": np.average(within_values, weights=within_weights),
    })

overall = pd.DataFrame(rows).set_index("model")
print(f"random-model PR-AUC would be {y.mean():.3f}")
display(overall)
overall.to_csv(REPORT_DIR / "08c_overall_comparison.csv")

random-model PR-AUC would be 0.228


,pr_auc,roc_auc,within_product_roc_auc
model,,,
B1 product rate,0.345107,0.691593,0.500000
B4 champion,0.427525,0.751925,0.677722
distilbert (transformer),0.448553,0.769966,0.705257
blend B4 + distilbert,0.439567,0.769368,0.707992
roberta (transformer),0.458919,0.773398,0.708887
blend B4 + roberta,0.446917,0.771815,0.712047


## 2. Where templates cannot help
A model can look good by recognising templates and company names. We re-score on two subsets that avoid that: narratives **without repeated wording**,
and the large **ordinary-stories topic** (the biggest topic from notebook 07).

**What this does:** flags repeated wording (a narrative whose first 150 characters appear 5 or more times in train plus validation), as in notebook 07b.

In [20]:
df["opening"] = df["text"].str.slice(0, OPENING_CHARS)
opening_counts = df["opening"].map(df["opening"].value_counts())
df["repeated_wording"] = opening_counts >= MIN_REPEATS
valid["repeated_wording"] = df.loc[is_valid, "repeated_wording"].to_numpy()
print(f"repeated wording: {valid['repeated_wording'].mean():.1%} of validation complaints")

repeated wording: 17.2% of validation complaints


**What this does:** finds each complaint's main topic with the saved topic model from notebook 07, and picks the biggest topic as the "ordinary stories" group.

In [21]:
topic_bundle = joblib.load(MODEL_DIR / "07_topic_model.joblib")
topic_weights = topic_bundle["nmf"].transform(topic_bundle["vectorizer"].transform(df["text"]))
df["topic"] = np.where(topic_weights.sum(axis=1) > 0, topic_weights.argmax(axis=1), -1)
valid["topic"] = df.loc[is_valid, "topic"].to_numpy()

organic_topic = df[~is_valid]["topic"].value_counts().idxmax()
print("ordinary-stories topic:", topic_bundle["topic_labels"][organic_topic])

ordinary-stories topic: T2: general bank/card/money stories


**What this does:** computes PR-AUC and ROC-AUC of every model on each subset.

In [22]:
subsets = {
    "all validation": np.ones(len(valid), dtype=bool),
    "non-repeated wording": (~valid["repeated_wording"]).to_numpy(),
    "ordinary-stories topic": (valid["topic"] == organic_topic).to_numpy(),
}

rows = []
for subset_name, mask in subsets.items():
    y_sub = y[mask]
    if mask.sum() < MIN_N_AUC or y_sub.min() == y_sub.max():
        continue
    for model_name, column in model_columns.items():
        scores = valid.loc[mask, column].to_numpy()
        rows.append({"subset": subset_name, "complaints": int(mask.sum()), "relief_rate": y_sub.mean(), "model": model_name,
                     "pr_auc": average_precision_score(y_sub, scores), "roc_auc": roc_auc_score(y_sub, scores)})

by_subset = pd.DataFrame(rows)
display(by_subset.pivot(index="model", columns="subset", values="pr_auc").round(4))
display(by_subset.pivot(index="model", columns="subset", values="roc_auc").round(4))
by_subset.to_csv(REPORT_DIR / "08c_by_subset.csv", index=False)

subset,all validation,non-repeated wording,ordinary-stories topic
model,,,
B1 product rate,0.3451,0.3083,0.2352
B4 champion,0.4275,0.4239,0.3678
blend B4 + distilbert,0.4396,0.4509,0.3863
blend B4 + roberta,0.4469,0.4536,0.3923
distilbert (transformer),0.4486,0.4599,0.3698
roberta (transformer),0.4589,0.4641,0.3805


subset,all validation,non-repeated wording,ordinary-stories topic
model,,,
B1 product rate,0.6916,0.6865,0.6646
B4 champion,0.7519,0.7627,0.7599
blend B4 + distilbert,0.7694,0.7866,0.7778
blend B4 + roberta,0.7718,0.7881,0.7827
distilbert (transformer),0.7700,0.7853,0.7680
roberta (transformer),0.7734,0.7868,0.7765


## 3. Is any gain real? Bootstrap and the decision
We resample the validation complaints 300 times with replacement and, each time, recompute the **gain over B4** in PR-AUC.
If the 95% interval of the gain is entirely above 0 *and* the average gain is at least +0.01, the rule says the new model wins.

**What this does:** runs the 300 resamples and stores, for every transformer and every blend, its PR-AUC gain over B4. This is the slowest cell
(about a minute per model).

In [23]:
rng = np.random.default_rng(SEED)
n = len(valid)
candidate_columns = [column for name, column in model_columns.items() if column not in ("p_b1", "p_b4")]
gains = {column: [] for column in candidate_columns}

for repeat in range(N_BOOT):
    pick = rng.integers(0, n, n)
    y_boot = y[pick]
    if y_boot.min() == y_boot.max():
        continue
    base = average_precision_score(y_boot, valid["p_b4"].to_numpy()[pick])
    for column in candidate_columns:
        gains[column].append(average_precision_score(y_boot, valid[column].to_numpy()[pick]) - base)

print("resamples used:", len(gains[candidate_columns[0]]))

resamples used: 300


**What this does:** turns the resamples into intervals. Each interval is **widened** according to how many comparisons we make (5% divided by that
number), then the rule is applied: the lower end must be above 0 and the average gain at least +0.01.

In [24]:
number_of_comparisons = len(candidate_columns)
tail = (5.0 / number_of_comparisons) / 2   # percent in each tail
print(f"{number_of_comparisons} comparisons -> each interval covers {100 - 2 * tail:.1f}% instead of 95%")

column_to_name = {column: name for name, column in model_columns.items()}
rows = []
for column in candidate_columns:
    values = np.array(gains[column])
    low, high = np.percentile(values, [tail, 100 - tail])
    meets_rule = bool(low > 0 and values.mean() >= MIN_GAIN)
    rows.append({"comparison": column_to_name[column] + " minus B4", "mean_pr_auc_gain": values.mean(),
                 "ci_low": low, "ci_high": high, "meets_rule": meets_rule})

verdict = pd.DataFrame(rows)
display(verdict)
verdict.to_csv(REPORT_DIR / "08c_bootstrap_verdict.csv", index=False)

if verdict["meets_rule"].any():
    winner = verdict[verdict["meets_rule"]].sort_values("mean_pr_auc_gain", ascending=False).iloc[0]["comparison"]
    print("RULE MET by:", winner, "-> it can become the new champion (its probabilities must be re-calibrated first).")
else:
    print("RULE NOT MET: B4 stays champion.")

4 comparisons -> each interval covers 98.8% instead of 95%


,comparison,mean_pr_auc_gain,ci_low,ci_high,meets_rule
0,distilbert (transformer) minus B4,0.021290,0.008948,0.033184,True
1,blend B4 + distilbert minus B4,0.011962,0.004362,0.019275,True
2,roberta (transformer) minus B4,0.031509,0.018797,0.042864,True
3,blend B4 + roberta minus B4,0.019286,0.012318,0.025563,True


RULE MET by: roberta (transformer) minus B4 -> it can become the new champion (its probabilities must be re-calibrated first).


## 4. Cost: why accuracy is not the only criterion
| | B4 (TF-IDF + logistic) | transformer |
|---|---|---|
| Training | about 1 minute on a laptop CPU | 20 to 90 minutes on a GPU |
| Prediction | milliseconds, CPU only | needs a GPU, or a slow CPU, for real-time use |
| Size | a few MB | 250 MB to 500 MB |
| Explanation | read the coefficients directly | needs extra tools (attention maps, SHAP) |

**Your conclusion (write it after reading the tables):** if the rule is not met, say so plainly. "A fine-tuned transformer did not beat a tuned linear model
on this task, so I shipped the simpler one" is a strong, honest result.

## How to explain this in an interview
> "I fine-tuned DistilBERT and RoBERTa on a free GPU and compared them with my champion under a rule written in advance: a model had to beat the champion by at least 0.01 PR-AUC
> with a bootstrap interval above zero, widened for the number of models I tried. I also scored both models on narratives without repeated wording and on the ordinary-stories topic, so templates could not
> flatter either model, and I weighed accuracy against cost, speed and explainability."